# DDIVF-HMM Pairs Trading Strategy

### An implementation of *"A Novel Algorithmic Trading Strategy using Hidden Markov Model for Kalman Filtering Innovations"*
Johnson-Skinner, Liang, Yu & Morariu — **IEEE COMPSAC 2021**
DOI: [10.1109/COMPSAC51774.2021.00264](https://doi.org/10.1109/COMPSAC51774.2021.00264)

---

## What this notebook does

This notebook reproduces, end-to-end, a **statistical arbitrage (pairs trading) strategy** that combines three ingredients:

1. **Kalman Filter (KF)** — tracks a *time-varying hedge ratio* between two cointegrated stocks.
2. **DDIVF (Data-Driven Innovation Volatility Forecast)** — a robust, non-parametric way to forecast the volatility of the KF's prediction errors ("innovations"), instead of relying on the KF's own (often poor) internal variance estimate.
3. **Hidden Markov Model (HMM)** — detects hidden "market regimes" (e.g. calm vs. volatile) directly from the innovation sequence, and lets the trading strategy use a **different signal threshold per regime**.

The core intuition: a pairs-trade spread has a natural equilibrium. When it drifts too far from equilibrium, we bet on reversion (short the expensive leg, long the cheap leg). The trick is estimating that equilibrium *dynamically* (via the Kalman filter), knowing how far is "too far" *without assuming Gaussian noise* (via DDIVF), and being more/less aggressive depending on whether the market is calm or turbulent (via the HMM).

## Structure

| Section | What happens |
|---|---|
| 1 | Theory recap: state-space model for pairs trading |
| 2 | Data: download two cointegrated stocks |
| 3 | Cointegration tests (Engle-Granger, Johansen) |
| 4 | Kalman Filter → dynamic hedge ratio & innovations |
| 5 | DDIVF → robust rolling volatility forecast of innovations |
| 6 | Baseline strategy: DDIVF-only pairs trading + backtest |
| 7 | HMM on innovations → regime detection |
| 8 | DDIVF-HMM strategy: regime-dependent thresholds + backtest |
| 9 | Results comparison (Buy & Hold vs. DDIVF vs. DDIVF-HMM) |
| 10 | Conclusion & ideas for extension |

> **Note on reproducibility:** this notebook downloads live data from Yahoo Finance via `yfinance`. If you're running it somewhere without internet access, it automatically falls back to a **synthetic cointegrated pair** with realistic (fat-tailed, regime-switching) noise, so every cell still runs and produces meaningful output.


## 1. Theory: the state-space model behind pairs trading

Given two cointegrated price series $P_{1,t}$ and $P_{2,t}$, the paper models the relationship with a **linear Gaussian state-space model**:

**State equation** (the hedge ratio evolves as a random walk — it's allowed to drift slowly over time):
$$\beta_t = \beta_{t-1} + v_t, \qquad v_t \sim (0, \Sigma_v)$$

**Observation equation:**
$$y_t = A_t \beta_t + \varepsilon_t, \qquad \varepsilon_t \sim (0, \sigma_\varepsilon^2)$$

where $y_t = P_{1,t}$ and $A_t = (1, P_{2,t})$, so $\beta_t = (\beta_{0,t}, \beta_{1,t})'$ is an intercept and a *time-varying hedge ratio*.

The **Kalman Filter** recursively computes the best (minimum-MSE) estimate $\hat\beta_{t|t}$ given data up to time $t$. At each step it also produces the **innovation**:

$$\nu_t = y_t - A_t \hat\beta_{t|t-1}$$

— i.e. the one-step-ahead prediction error. This is the key signal: $\nu_t$ tells us how far the observed spread is from what the model expected, and large innovations (relative to their typical size) are exactly what a mean-reversion strategy wants to trade on.

The problem: the "typical size" of $\nu_t$, $\sqrt{Q_t}$, comes straight out of the filter's internal covariance recursion — but the paper shows (citing Thavaneswaran et al. 2019) that this is **not a statistically efficient estimator** of innovation volatility, especially when the true innovations are non-normal / fat-tailed (very common in financial returns). That motivates **DDIVF** in Section 5.


In [ ]:
# --- Imports & configuration ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsmodels.tsa.stattools import coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from hmmlearn.hmm import GaussianHMM

np.random.seed(42)
plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ---- Choose your pair here ----
TICKER_1 = "KO"    # Coca-Cola  (plays the role of P1 / AOS in the paper)
TICKER_2 = "PEP"   # PepsiCo    (plays the role of P2 / DUK in the paper)
START_DATE = "2018-01-01"
END_DATE   = "2021-01-01"
TRAIN_END  = "2020-02-09"   # matches the paper's train/test split date


## 2. Data

We download daily adjusted close prices for two historically cointegrated stocks. `KO`/`PEP` (Coca-Cola / PepsiCo) is a classic textbook pairs-trading example — both are large, stable consumer-staples companies whose prices tend to move together over the long run, similar in spirit to the AOS/DUK utility-sector pair used in the paper.

If there's no internet access in this environment, the cell below **automatically falls back** to a simulated cointegrated pair (random-walk common factor + a slowly time-varying hedge ratio + Student-*t* noise), so the rest of the notebook still runs and demonstrates the full pipeline correctly.


In [ ]:
def simulate_cointegrated_pair(n=755, seed=42):
    '''Fallback synthetic data generator: two cointegrated series with a
    slowly time-varying hedge ratio and fat-tailed (Student-t) noise, so the
    notebook's algorithms have realistic, non-Gaussian input even offline.'''
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2018-01-01", periods=n)

    p2 = 50 + np.cumsum(rng.normal(0, 0.5, n))
    beta_true = 1.1 + np.cumsum(rng.normal(0, 0.004, n))       # drifting hedge ratio
    # inject a volatile regime (like March 2020) partway through
    vol = np.ones(n)
    vol[int(n*0.75):int(n*0.80)] = 4.0
    noise = rng.standard_t(4, n) * 0.9 * vol
    p1 = 15 + beta_true * p2 + noise

    df = pd.DataFrame({"P1": p1, "P2": p2}, index=dates)
    return df


def fetch_price_data(t1, t2, start, end):
    try:
        import yfinance as yf
        raw = yf.download([t1, t2], start=start, end=end, progress=False, auto_adjust=True)["Close"]
        raw = raw.dropna()
        if raw.empty or raw.shape[0] < 200:
            raise ValueError("Insufficient data returned from Yahoo Finance")
        df = raw.rename(columns={t1: "P1", t2: "P2"})[["P1", "P2"]]
        print(f"Downloaded {len(df)} trading days for {t1}/{t2} from Yahoo Finance.")
        return df, True
    except Exception as e:
        print(f"[Falling back to synthetic data] Could not fetch live data ({e}).")
        return simulate_cointegrated_pair(), False


prices, using_real_data = fetch_price_data(TICKER_1, TICKER_2, START_DATE, END_DATE)
label1, label2 = (TICKER_1, TICKER_2) if using_real_data else ("Sim-P1", "Sim-P2")

train_mask = prices.index <= TRAIN_END
print(f"Train: {train_mask.sum()} days | Test: {(~train_mask).sum()} days")
prices.tail()


In [ ]:
fig, ax = plt.subplots()
ax.plot(prices.index, prices["P1"], label=label1)
ax.plot(prices.index, prices["P2"], label=label2)
split_date = prices.index[train_mask][-1]
ax.axvline(split_date, color="black", linestyle="--", linewidth=1, label="train/test split")
ax.set_title("Daily Adjusted Close Prices")
ax.set_ylabel("Price")
ax.legend()
plt.tight_layout()
plt.show()


## 3. Cointegration tests

Pairs trading only makes sense if the two series are genuinely cointegrated — i.e. some linear combination of them is stationary (mean-reverting), even though each series individually is a non-stationary random walk. The paper checks this with both the **Engle-Granger** test and the **Johansen** test; we do the same here on the training window.

- **Engle-Granger**: regress $P_{1,t}$ on $P_{2,t}$, then test the residual for stationarity (ADF test). A small p-value (< 0.05) supports cointegration.
- **Johansen**: a multivariate likelihood-ratio test that also works when there are more than two series; here we just confirm the trace statistic exceeds its critical value at rank 0.


In [ ]:
train = prices.loc[train_mask]
test = prices.loc[~train_mask]

# Engle-Granger
eg_stat, eg_pvalue, _ = coint(train["P1"], train["P2"])
print(f"Engle-Granger test statistic: {eg_stat:.3f}, p-value: {eg_pvalue:.4f}")
print("=> Cointegrated (p < 0.05)" if eg_pvalue < 0.05 else "=> No strong evidence of cointegration")

# Johansen
johansen_result = coint_johansen(train[["P1", "P2"]].values, det_order=0, k_ar_diff=1)
trace_stat = johansen_result.lr1[0]
crit_95 = johansen_result.cvt[0, 1]
print(f"\nJohansen trace statistic (rank=0): {trace_stat:.3f}, 95% critical value: {crit_95:.3f}")
print("=> Reject rank-0 null, i.e. cointegrated" if trace_stat > crit_95 else "=> Cannot reject rank-0 null")


## 4. Kalman Filter — dynamic hedge ratio & innovations (Algorithm 2)

This implements **Algorithm 2** from the paper directly:

1. **Predict**: $\hat\beta_{t|t-1} = \hat\beta_{t-1|t-1}$, $\;P_{t|t-1} = P_{t-1|t-1} + \Sigma_v$
2. **Innovation**: $\nu_t = y_t - A_t \hat\beta_{t|t-1}$, with variance $Q_t = A_t P_{t|t-1} A_t' + \sigma_\varepsilon^2$
3. **Update**: $\hat\beta_{t|t} = \hat\beta_{t|t-1} + P_{t|t-1} A_t' Q_t^{-1} \nu_t$, $\;P_{t|t} = (I - P_{t|t-1}A_t'Q_t^{-1}A_t)\,P_{t|t-1}$

We run this over the **entire** price history (train + test), exactly as the paper does, since the filter itself is causal (only ever uses information up to time $t$) — the train/test split only matters later, when we *fit thresholds* on the training portion and *evaluate* on the test portion.


In [ ]:
def kalman_filter_hedge_ratio(y, A, sigma_eps2=0.1, sigma_v2=1e-4):
    '''
    Algorithm 2: Kalman filter for a random-walk hedge ratio beta_t in the
    state-space model  beta_t = beta_{t-1} + v_t ,  y_t = A_t beta_t + eps_t.

    Parameters
    ----------
    y : (n,) array -- observed process, y_t = P1_t
    A : (n, m) array -- predictors, A_t = (1, P2_t)
    sigma_eps2 : observation noise variance
    sigma_v2   : state noise variance (isotropic: Sigma_v = sigma_v2 * I)

    Returns
    -------
    beta : (n, m) filtered state estimates beta_hat_{t|t}
    nu   : (n,) innovations
    Q    : (n,) innovation variances (the KF's own -- not yet DDIVF-adjusted)
    '''
    n, m = A.shape
    beta = np.zeros((n, m))
    nu = np.zeros(n)
    Q = np.zeros(n)

    beta_prev = np.zeros(m)
    P_prev = np.eye(m)
    Sigma_v = np.eye(m) * sigma_v2

    for t in range(n):
        # --- Prediction ---
        beta_pred = beta_prev
        P_pred = P_prev + Sigma_v
        y_pred = A[t] @ beta_pred

        # --- Innovation ---
        innovation = y[t] - y_pred
        Q_t = A[t] @ P_pred @ A[t].T + sigma_eps2

        # --- Update ---
        K_t = P_pred @ A[t].T / Q_t
        beta_t = beta_pred + K_t * innovation
        P_t = (np.eye(m) - np.outer(K_t, A[t])) @ P_pred

        beta[t], nu[t], Q[t] = beta_t, innovation, Q_t
        beta_prev, P_prev = beta_t, P_t

    return beta, nu, Q


y_all = prices["P1"].values
A_all = np.column_stack([np.ones(len(prices)), prices["P2"].values])

beta, nu, Q = kalman_filter_hedge_ratio(y_all, A_all)

kf_df = pd.DataFrame(
    {"beta0": beta[:, 0], "beta1": beta[:, 1], "innovation": nu, "Q": Q},
    index=prices.index,
)
kf_df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(kf_df.index, kf_df["beta1"], color="tab:purple")
axes[0].axvline(split_date, color="black", linestyle="--", linewidth=1)
axes[0].set_title(f"Filtered dynamic hedge ratio  $\\hat\\beta_{{1,t}}$  ({label1} on {label2})")
axes[0].set_ylabel(r"$\hat\beta_{1,t}$")

axes[1].plot(kf_df.index, kf_df["innovation"], color="tab:blue", linewidth=0.8)
axes[1].axvline(split_date, color="black", linestyle="--", linewidth=1)
axes[1].axhline(0, color="grey", linewidth=0.8)
axes[1].set_title(r"Kalman filter innovations $\nu_t$")
axes[1].set_ylabel(r"$\nu_t$")
plt.tight_layout()
plt.show()


## 5. DDIVF — Data-Driven Innovation Volatility Forecast (Algorithm 1)

Rather than trust $\sqrt{Q_t}$ from the Kalman filter, we forecast the volatility of $\nu_t$ directly from its recent history using a **DD-EWMA** (data-driven exponentially weighted moving average):

$$\hat\sigma_t = (1-\alpha)\,\hat\sigma_{t-1} + \alpha \cdot \frac{|\nu_{t-1} - \bar\nu|}{\hat\rho_\nu}, \qquad 0 < \alpha < 1$$

where $\hat\rho_\nu = \mathrm{Corr}(\nu_t - \bar\nu,\ \mathrm{sign}(\nu_t - \bar\nu))$ is the **estimated sign correlation** — a robust quantity that adapts the EWMA to the actual (possibly non-normal / heavy-tailed) shape of the innovation distribution, rather than assuming Gaussian errors.

**Algorithm 1**, implemented below:
1. Over a rolling window of the last $k$ innovations, compute $\hat\rho_\nu$ and the volatility series $V_s = |\nu_s - \bar\nu| / \hat\rho_\nu$.
2. Grid-search $\alpha \in (0.01, 0.5)$ to find the value that **minimizes the one-step-ahead forecast error sum of squares (FESS)**.
3. Use the resulting smoothed value as the volatility forecast $\hat\sigma^{DD}_t$ for the *next* innovation.

This runs on a **rolling window** (paper uses $k=100$ days), producing one forecast per day, always using only past information (no look-ahead).


In [ ]:
def ddivf_forecast(nu_window, alpha_grid=None):
    '''
    Algorithm 1: given a window of past innovations, find the optimal DD-EWMA
    smoothing constant alpha (by minimizing one-step-ahead FESS) and return
    the resulting one-step-ahead volatility forecast.
    '''
    if alpha_grid is None:
        alpha_grid = np.arange(0.01, 0.51, 0.01)

    nu = np.asarray(nu_window)
    nu_bar = nu.mean()
    dev = nu - nu_bar
    sgn = np.sign(dev)

    # estimated sign correlation rho_hat_nu; guard against degenerate windows
    if np.std(sgn) == 0 or np.std(dev) == 0:
        rho_hat = 1e-6
    else:
        rho_hat = np.corrcoef(dev, sgn)[0, 1]
        if abs(rho_hat) < 1e-6:
            rho_hat = 1e-6

    V = np.abs(dev) / abs(rho_hat)
    l = max(5, len(V) // 5)  # warm-up length before scoring FESS

    best_alpha, best_fess = alpha_grid[0], np.inf
    for a in alpha_grid:
        S = np.zeros(len(V))
        S[0] = V[:l].mean()
        for s in range(1, len(V)):
            S[s] = a * V[s] + (1 - a) * S[s - 1]
        fess = np.sum((V[l:] - S[l - 1:-1]) ** 2)
        if fess < best_fess:
            best_fess, best_alpha = fess, a

    # recompute smoothed series with the chosen alpha to get today's forecast
    S = np.zeros(len(V))
    S[0] = V[:l].mean()
    for s in range(1, len(V)):
        S[s] = best_alpha * V[s] + (1 - best_alpha) * S[s - 1]

    return best_alpha, S[-1]


def rolling_ddivf(nu, k=100):
    '''Produces a full-length array of one-day-ahead DDIVF forecasts,
    NaN for the first k days where there isn't enough history yet.'''
    n = len(nu)
    sigma_dd = np.full(n, np.nan)
    for t in range(k, n):
        window = nu[t - k:t]
        _, sigma_dd[t] = ddivf_forecast(window)
    return sigma_dd


K_WINDOW = 100
sigma_dd = rolling_ddivf(kf_df["innovation"].values, k=K_WINDOW)
kf_df["sigma_dd"] = sigma_dd
print(f"DDIVF computed for {kf_df['sigma_dd'].notna().sum()} of {len(kf_df)} days "
      f"(first {K_WINDOW} days used as the initial rolling window).")


## 6. Baseline strategy: DDIVF pairs trading (Algorithm 3, single threshold)

Now we implement the trading rule itself. Define the robust z-score $z_t = \nu_t / \hat\sigma^{DD}_t$ and compare it against a threshold $p$:

- **Sell** ($s_t=-1$, sell the "expensive" leg) when $\nu_t$ crosses **above** $p\cdot\hat\sigma^{DD}_t$ from below.
- **Buy** ($s_t=+1$, buy the "cheap" leg) when $\nu_t$ crosses **below** $-p\cdot\hat\sigma^{DD}_t$ from above.
- Otherwise, no new signal ($s_t = 0$; a signal typically triggers a position that's held until reversion / exit conditions).

Positions are sized as $1000\times$ the spread, split between the two legs using the filtered hedge ratio $\hat\beta_{t-1|t-1}$ (exactly as in Algorithm 3). We find the optimal threshold $p_{opt}$ by grid-searching $p \in [0.5, 2.5]$ in steps of $0.1$ on the **training data**, maximizing the annualized Sharpe ratio:

$$ASR = \sqrt{252} \cdot \frac{\text{mean}(profit_t)}{\text{sd}(profit_t)}$$

...then evaluate that fixed $p_{opt}$ **out-of-sample** on the test period.


In [ ]:
def backtest_strategy(beta, nu, sigma_dd, P1, P2, thresholds, k, state=None):
    '''
    Implements the trading-signal / P&L logic of Algorithm 3.

    thresholds : either a single float p (DDIVF-only strategy) or a dict
                 {state_id: p} for the regime-aware DDIVF-HMM strategy.
    state       : (n,) array of hidden states (required if thresholds is a dict)

    Returns a DataFrame with signals, positions and daily profit.
    '''
    n = len(nu)
    s = np.zeros(n)

    def p_at(t):
        if isinstance(thresholds, dict):
            st = state[t]
            return thresholds.get(st, np.nan) if st in thresholds else np.nan
        return thresholds

    for t in range(k + 1, n):
        if np.isnan(sigma_dd[t]) or np.isnan(sigma_dd[t - 1]):
            continue
        p_t, p_tm1 = p_at(t), p_at(t - 1)
        if p_t is None or p_tm1 is None or (isinstance(p_t, float) and np.isnan(p_t)):
            continue
        upper_t, upper_tm1 = p_t * sigma_dd[t], p_tm1 * sigma_dd[t - 1]
        lower_t, lower_tm1 = -p_t * sigma_dd[t], -p_tm1 * sigma_dd[t - 1]
        if nu[t] > upper_t and nu[t - 1] < upper_tm1:
            s[t] = -1
        elif nu[t] < lower_t and nu[t - 1] > lower_tm1:
            s[t] = 1

    beta1_lag = np.roll(beta[:, 1], 1)
    position_A = -1000 * beta1_lag * s
    position_y = 1000 * s

    dP2 = np.diff(P2, prepend=P2[0])
    dP1 = np.diff(P1, prepend=P1[0])
    profit = position_A * dP2 + position_y * dP1

    out = pd.DataFrame({"signal": s, "profit": profit})
    return out


def strategy_summary(profit, label="strategy"):
    cum_profit = profit.sum()
    asr = np.sqrt(252) * profit.mean() / profit.std() if profit.std() > 0 else 0.0
    return {"strategy": label, "ASR": round(asr, 2), "cumulative_profit": round(cum_profit, 2)}


def buy_and_hold_profit(P1, P2, mask):
    '''1000 units long P1, 1000 units short P2 (simple B/H benchmark), over the given mask.'''
    p1, p2 = P1[mask], P2[mask]
    return 1000 * (p1[-1] - p1[0]) - 1000 * (p2[-1] - p2[0])


P1_arr, P2_arr = prices["P1"].values, prices["P2"].values
train_mask = pd.Series(train_mask, index=prices.index)
train_idx = np.where(train_mask)[0]
test_idx = np.where(~train_mask)[0]
k = K_WINDOW

p_grid = np.arange(0.5, 2.51, 0.1)
best_p, best_asr = None, -np.inf
for p in p_grid:
    bt = backtest_strategy(beta[train_idx], kf_df["innovation"].values[train_idx],
                            kf_df["sigma_dd"].values[train_idx],
                            P1_arr[train_idx], P2_arr[train_idx], p, k)
    res = strategy_summary(bt["profit"])
    if res["ASR"] > best_asr:
        best_asr, best_p = res["ASR"], p

print(f"Optimal single threshold on training data: p_opt = {best_p:.2f}")


In [ ]:
# --- Train performance with p_opt ---
bt_train = backtest_strategy(beta[train_idx], kf_df["innovation"].values[train_idx],
                              kf_df["sigma_dd"].values[train_idx],
                              P1_arr[train_idx], P2_arr[train_idx], best_p, k)
bh_train = buy_and_hold_profit(P1_arr, P2_arr, train_mask)
res_train = strategy_summary(bt_train["profit"], "DDIVF (train)")
res_train["buy_hold_profit"] = round(bh_train, 2)

# --- Test performance with the SAME p_opt (out of sample) ---
bt_test = backtest_strategy(beta[test_idx], kf_df["innovation"].values[test_idx],
                             kf_df["sigma_dd"].values[test_idx],
                             P1_arr[test_idx], P2_arr[test_idx], best_p, k)
bh_test = buy_and_hold_profit(P1_arr, P2_arr, ~train_mask)
res_test = strategy_summary(bt_test["profit"], "DDIVF (test)")
res_test["buy_hold_profit"] = round(bh_test, 2)

ddivf_results = pd.DataFrame([res_train, res_test]).set_index("strategy")
ddivf_results


In [ ]:
fig, ax = plt.subplots()
train_dates = prices.index[train_idx]
ax.plot(train_dates, kf_df["innovation"].values[train_idx], label="Innovation", linewidth=0.8)
ax.plot(train_dates, best_p * kf_df["sigma_dd"].values[train_idx], label="Upper band", color="tab:orange")
ax.plot(train_dates, -best_p * kf_df["sigma_dd"].values[train_idx], label="Lower band", color="tab:green")
ax.axhline(0, color="grey", linewidth=0.6)
ax.set_title(f"DDIVF trading signals (training period), p_opt = {best_p:.2f}")
ax.legend()
plt.tight_layout()
plt.show()


## 7. Adding the HMM: regime-aware thresholds (Algorithm 3, DDIVF-HMM)

So far, one fixed threshold $p_{opt}$ is used for the *entire* history — but market conditions clearly aren't constant (think of March 2020). The paper's key idea: fit a **Hidden Markov Model** on the innovation sequence $\nu_t$ to discover latent regimes (e.g. "calm" vs. "volatile"), then let the trading threshold **depend on the current hidden state** $S_t$:

$$p^{HMM}_{opt}[S_t], \qquad S_t \in \{1, \dots, K\}$$

We use `hmmlearn`'s `GaussianHMM` to fit $K$-state models directly on the innovations (this plays the role of the paper's EM-estimated transition/emission parameters), decode the most likely hidden state at each $t$ with the Viterbi algorithm, then **grid-search a separate optimal threshold for each state** on the training data (maximizing ASR), exactly mirroring Algorithm 3's brute-force search over the matrix $p^{HMM}_{P^K,K}$.

We try **2 hidden states** first, then **3**, matching the paper's Section III-B/III-C.


In [ ]:
def fit_hmm_states(nu, valid_mask, n_states, seed=0):
    '''Fit a Gaussian HMM on valid (non-NaN) innovations and decode states
    for the full series (state = -1 where DDIVF wasn't yet available).'''
    obs = nu[valid_mask].reshape(-1, 1)
    model = GaussianHMM(n_components=n_states, covariance_type="diag",
                         n_iter=500, random_state=seed)
    model.fit(obs)
    decoded = model.predict(obs)

    states = np.full(len(nu), -1)
    states[valid_mask] = decoded
    return model, states


def optimal_thresholds_per_state(beta, nu, sigma_dd, states, P1, P2, idx, n_states, p_grid, k):
    '''Brute-force search (Algorithm 3) for the per-state threshold vector
    that maximizes ASR on the given (training) index range.'''
    from itertools import product
    best_asr, best_p_vec = -np.inf, None
    for combo in product(p_grid, repeat=n_states):
        p_vec = {s: combo[s] for s in range(n_states)}
        bt = backtest_strategy(beta[idx], nu[idx], sigma_dd[idx], P1[idx], P2[idx],
                                p_vec, k, state=states[idx])
        res = strategy_summary(bt["profit"])
        if res["ASR"] > best_asr:
            best_asr, best_p_vec = res["ASR"], p_vec
    return best_p_vec, best_asr


valid_mask_all = ~np.isnan(kf_df["sigma_dd"].values) & train_mask.values
nu_arr = kf_df["innovation"].values

# a coarser grid for the multi-dimensional brute-force search (keeps runtime reasonable)
p_grid_hmm = np.round(np.arange(0.5, 2.51, 0.2), 2)


### 7a. Two-state HMM

In [ ]:
hmm2, states2 = fit_hmm_states(nu_arr, valid_mask_all, n_states=2)
# extend decoding to the full series (train+test) using the fitted model, for consistent state labels
valid_full = ~np.isnan(kf_df["sigma_dd"].values)
states2_full = np.full(len(nu_arr), -1)
states2_full[valid_full] = hmm2.predict(nu_arr[valid_full].reshape(-1, 1))

print("HMM (2-state) emission means:", np.round(hmm2.means_.ravel(), 3))
print("HMM (2-state) state counts (train):", np.bincount(states2[states2 >= 0]))

p_vec_2, train_asr_2 = optimal_thresholds_per_state(
    beta, nu_arr, kf_df["sigma_dd"].values, states2_full, P1_arr, P2_arr,
    train_idx, n_states=2, p_grid=p_grid_hmm, k=k
)
print("Optimal per-state thresholds (2-state HMM):", p_vec_2)


In [ ]:
fig, ax = plt.subplots()
for st, color in zip(range(2), ["tab:red", "gold"]):
    mask = (states2_full == st) & train_mask.values
    ax.scatter(prices.index[mask], kf_df["innovation"].values[mask], s=8, color=color, label=f"State {st}")
ax.plot(train_dates, p_vec_2[0] * kf_df["sigma_dd"].values[train_idx], color="grey", alpha=0.4)
ax.set_title("Innovations colored by hidden HMM state (2-state model, training period)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
bt_train_hmm2 = backtest_strategy(beta[train_idx], nu_arr[train_idx], kf_df["sigma_dd"].values[train_idx],
                                   P1_arr[train_idx], P2_arr[train_idx], p_vec_2, k, state=states2_full[train_idx])
res_train_hmm2 = strategy_summary(bt_train_hmm2["profit"], "DDIVF-HMM 2-state (train)")
res_train_hmm2["buy_hold_profit"] = round(bh_train, 2)

bt_test_hmm2 = backtest_strategy(beta[test_idx], nu_arr[test_idx], kf_df["sigma_dd"].values[test_idx],
                                  P1_arr[test_idx], P2_arr[test_idx], p_vec_2, k, state=states2_full[test_idx])
res_test_hmm2 = strategy_summary(bt_test_hmm2["profit"], "DDIVF-HMM 2-state (test)")
res_test_hmm2["buy_hold_profit"] = round(bh_test, 2)

hmm2_results = pd.DataFrame([res_train_hmm2, res_test_hmm2]).set_index("strategy")
hmm2_results


### 7b. Three-state HMM

The paper notes the number of hidden states is a hyperparameter, chosen on the training data by whichever gives the best Sharpe ratio, and cautions against going beyond 3 states to avoid overfitting. We repeat the same procedure with $K=3$.


In [ ]:
hmm3, states3 = fit_hmm_states(nu_arr, valid_mask_all, n_states=3)
states3_full = np.full(len(nu_arr), -1)
states3_full[valid_full] = hmm3.predict(nu_arr[valid_full].reshape(-1, 1))

print("HMM (3-state) emission means:", np.round(hmm3.means_.ravel(), 3))
print("HMM (3-state) state counts (train):", np.bincount(states3[states3 >= 0]))

p_vec_3, train_asr_3 = optimal_thresholds_per_state(
    beta, nu_arr, kf_df["sigma_dd"].values, states3_full, P1_arr, P2_arr,
    train_idx, n_states=3, p_grid=p_grid_hmm, k=k
)
print("Optimal per-state thresholds (3-state HMM):", p_vec_3)


In [ ]:
bt_train_hmm3 = backtest_strategy(beta[train_idx], nu_arr[train_idx], kf_df["sigma_dd"].values[train_idx],
                                   P1_arr[train_idx], P2_arr[train_idx], p_vec_3, k, state=states3_full[train_idx])
res_train_hmm3 = strategy_summary(bt_train_hmm3["profit"], "DDIVF-HMM 3-state (train)")
res_train_hmm3["buy_hold_profit"] = round(bh_train, 2)

bt_test_hmm3 = backtest_strategy(beta[test_idx], nu_arr[test_idx], kf_df["sigma_dd"].values[test_idx],
                                  P1_arr[test_idx], P2_arr[test_idx], p_vec_3, k, state=states3_full[test_idx])
res_test_hmm3 = strategy_summary(bt_test_hmm3["profit"], "DDIVF-HMM 3-state (test)")
res_test_hmm3["buy_hold_profit"] = round(bh_test, 2)

hmm3_results = pd.DataFrame([res_train_hmm3, res_test_hmm3]).set_index("strategy")
hmm3_results


## 8. Results comparison

Putting all strategies side by side, on both the training window (in-sample fit) and the test window (out-of-sample / genuine performance test) — mirroring Tables I–VI in the paper.


In [ ]:
all_results = pd.concat([ddivf_results, hmm2_results, hmm3_results])
all_results


In [ ]:
test_rows = all_results[all_results.index.str.contains("test")]
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(test_rows.index, test_rows["cumulative_profit"], color=["tab:blue", "tab:orange", "tab:green"])
ax.axhline(bh_test, color="grey", linestyle="--", label=f"Buy & Hold = {bh_test:,.0f}")
ax.set_ylabel("Cumulative profit ($)")
ax.set_title("Out-of-sample (test period) cumulative profit by strategy")
ax.legend()
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


## 9. Conclusion & ideas for extension

**What we reproduced:**
- A Kalman filter that tracks a *time-varying* hedge ratio between two cointegrated assets and produces one-step-ahead prediction errors (innovations).
- DDIVF: a robust, distribution-free volatility forecast for those innovations, avoiding the KF's own (statistically inefficient) variance estimate.
- A baseline mean-reversion trading rule using a single global threshold.
- An HMM-based extension that detects hidden volatility regimes in the innovations and lets the trading threshold adapt per regime — generally allowing **wider bands (fewer, more selective trades) in volatile regimes** and **tighter bands (more trades) in calm regimes**, which matches basic trading intuition.

**Caveats to keep in mind before using anything like this with real capital:**
- No transaction costs, slippage, or borrowing costs for the short leg are modeled here — these can matter a lot for a strategy that trades frequently.
- The brute-force threshold search is fit and evaluated on the *same* fixed train/test split; a proper walk-forward / rolling re-estimation would be more robust to regime drift.
- HMM state labels are not guaranteed to be stable across different random seeds / refits — always sanity-check by inspecting the emission means (as we do above) before trusting "state 0" vs. "state 1" comparisons.
- Cointegration itself can break down over time; production systems should keep re-testing it (as the paper mentions doing "regularly").

**Natural next steps:**
- Add transaction costs and position-sizing constraints to the backtest.
- Try Student-*t* emission distributions in the HMM instead of Gaussian, given the DDIVF motivation of non-normal innovations.
- Walk-forward re-fitting of both the HMM and the thresholds instead of a single static train/test split.
- Test on a broader universe of pairs / sectors to see how consistently the HMM layer adds value over DDIVF alone.

---

**Reference**

Johnson-Skinner, E., Liang, Y., Yu, N., & Morariu, A. (2021). *A Novel Algorithmic Trading Strategy using Hidden Markov Model for Kalman Filtering Innovations.* 2021 IEEE 45th Annual Computers, Software, and Applications Conference (COMPSAC), 1766–1771. https://doi.org/10.1109/COMPSAC51774.2021.00264
